# Lab 2 — Knowledge & Grounding (🟡 Builder rail)

> 🟡 **Builder rail.** Run the cells top to bottom. The lines marked `# 👉` are the parts you wire up; the `# TODO (bonus)` cell is optional. Same checkpoint as the other rails.

**Goal:** give Care Pal a private **knowledge base** so every education answer is *grounded* — it cites a real `healthhub.sg` article instead of making things up. This is called **RAG** (Retrieval‑Augmented Generation).

> ⚠️ **Prerequisite:** the curated HealthHub docs must already be in `../knowledge/healthhub-discharge-pack/` (heart / kidney / liver self‑care files). If that folder is empty there's nothing to cite and Cell 3 will error — see that folder's `README`.

### What's new versus Lab 1? (the tech stack, in plain terms)

In Lab 1 the agent answered from the model's own knowledge. Here we **attach a document index** so the model retrieves real passages before answering.

| Building block | What it is | How it interacts with Microsoft Foundry |
|----------------|-----------|-----------------------------------------|
| **Vector store** | A searchable index of your documents. Foundry chunks each file, turns it into embeddings, and stores it. | Built via the OpenAI‑compatible client: `openai.vector_stores.create(...)` then upload each file. |
| **File‑search tool** (`FileSearchTool`) | A **tool** you bolt onto the agent that lets it search that vector store at answer time. | Passed in `tools=[...]` on the agent definition — Foundry runs the retrieval automatically. |
| **Grounding instructions** | Extra rules telling the agent to answer *from the knowledge base* and fill `source_labels` / `source_urls` — and to decline rather than invent when unsupported. | Layered on top of the Lab 1 triage instructions. |
| **Citations** | The `source_labels` + `source_urls` fields in the triage JSON, now populated with real HealthHub links. | Proof the answer came from a trusted source, not the model's imagination. |

> 📚 **Where this pattern comes from (optional reading):** the *file‑search quickstart* and `samples/agents/tools/sample_agent_file_search.py` in the `azure-ai-projects` 2.x samples. The `common/` helpers already apply it for you.


## Cell 1 — Make the shared helpers importable

Same setup step as Lab 1: add this notebook's folder (`assets/`) to Python's import path so the next cell can do `from common.carepal_common import ...`. It works whether you run this as a notebook or as the `python lab2_rag.py` script. Nothing touches Foundry yet — pure local setup.


In [1]:
import sys
import pathlib
_here = (pathlib.Path(globals()["__file__"]).resolve().parent
         if "__file__" in globals() else pathlib.Path.cwd())
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

## Cell 2 — Import the helpers and write the grounding instructions

This cell brings in the helpers and defines **how** the agent should use the knowledge base.

| Helper | What it does with Foundry |
|--------|---------------------------|
| `build_vector_store` | Uploads every file in the HealthHub pack and builds a **vector store** (the searchable index) via `openai.vector_stores.*`. Returns the store's id. |
| `file_search_tool` | Wraps that store id in a `FileSearchTool` so the agent can search it at answer time. |
| `make_triage_agent` | Same as Lab 1, but here we pass the grounding instructions **and** the file‑search tool. |
| `run_and_parse` | Sends a message to the agent and parses the returned triage JSON. |
| `text_of` / `cleanup` | Look up a test message; delete the agent afterwards. |
| `TRIAGE_INSTRUCTIONS` | The Lab 1 triage rules we build on top of. |

> 📄 **Where do the sample questions come from?** `text_of("diet_question")` doesn't hard‑code a string — it looks up the prompt by id in **[../prompts/test-prompts.json](../prompts/test-prompts.json)**, the single source of truth shared by every rail. For example, `diet_question` maps to *"What diet should my father follow after heart failure?"* along with its expected route and allowed citation hosts. Open that file to see all the prompt ids you can pass to `text_of(...)`.

`GROUNDING` is the key addition: it appends a paragraph to the Lab 1 instructions telling the agent to **ground education answers in the attached knowledge base**, fill `source_labels` / `source_urls` with the HealthHub articles it used, and — importantly — to **say it's unsure and route to `timely_review` rather than invent sources** when the pack doesn't cover the question.


In [2]:
import json
from common.carepal_common import (
    make_triage_agent,
    run_and_parse,
    build_vector_store,
    file_search_tool,
    text_of,
    cleanup,
    TRIAGE_INSTRUCTIONS,
)

GROUNDING = TRIAGE_INSTRUCTIONS + """
For education / self-care questions (route "education_navigation"), ground your reply in the attached
HealthHub knowledge base. Put the article titles in source_labels and their healthhub.sg URLs in
source_urls. If the knowledge base does not support an answer, say you are not sure and route to
"timely_review" - do NOT invent sources.
"""

## Cell 3 — Build the index, attach it, and check the answer is grounded

This is the heart of the lab — the full RAG round‑trip with Microsoft Foundry:

1. **Build the vector store** — `build_vector_store("healthhub-discharge-pack")` uploads the pack and returns a store id. This is where Foundry chunks + embeds your documents.
2. **Attach it as a tool** — `file_search_tool(vs_id)` wraps the store in a `FileSearchTool`.
3. **Create the grounded agent** — `make_triage_agent(instructions=GROUNDING, tools=[fs], structured=True)` provisions `carepal-jb` with the grounding rules **and** the retrieval tool.
4. **Ask the diet question** — `run_and_parse` sends it; Foundry searches the vector store, feeds the passages to the model, and returns the triage JSON.
5. **Validate grounding** — asserts `source_urls` is non‑empty **and** at least one URL is on the `healthhub.sg` host (the lab checkpoint).

The agent is **left running on purpose** so you can inspect it in the portal (Cell 4); you delete it in the final cleanup cell.

**Expected output:** the printed JSON has populated `source_labels` / `source_urls` with HealthHub links, followed by `Lab 2 passed ✅`.

> 💡 The `# TODO (bonus)` note is optional: send the un‑grounded LiverTone question (`unsupported_question`) and assert `source_urls` is **empty** and the route is `timely_review` — proving the agent declines instead of fabricating a citation.


In [3]:
# 👉 Build a vector store from the pack and attach it as a file-search tool.
vs_id = build_vector_store("healthhub-discharge-pack")
fs = file_search_tool(vs_id)

# 👉 Create the grounded agent (Lab 1 triage rules + grounding + the file-search tool).
# `agent` is kept as a top-level variable so you can inspect it in the Foundry
# portal and then delete it from the separate cleanup cell below.
agent = make_triage_agent(
    instructions=GROUNDING,
    tools=[fs],
    structured=True,
)
print(f"Created agent: {agent.name} (version {agent.version})")

out = run_and_parse(agent, text_of("diet_question"))
print(json.dumps(out, indent=2))
assert out["source_urls"], "expected at least one citation"
assert any("healthhub.sg" in u for u in out["source_urls"]), out["source_urls"]
print("Lab 2 passed ✅")

# TODO (bonus): send the LiverTone question (prompt id 'unsupported_question') and assert
# source_urls is EMPTY and route == 'timely_review' (the safe, no-fabrication behaviour).


Created agent: carepal-jb (version 5)
{
  "intent": "self_care_education",
  "risk_level": "low",
  "route": "education_navigation",
  "reply": "After heart failure, diet usually focuses on controlling salt and sometimes fluids to prevent fluid build\u2011up. Try to keep salt low: cook with herbs, garlic, ginger, lemon or pepper instead of salt. Limit high\u2011salt foods like processed meats, canned soups, instant noodles, salted snacks, and salty sauces (for example soy or oyster sauce). Choose lower\u2011sodium options when reading food labels. If his care team gave a daily fluid limit, include drinks, soup, porridge, ice and jelly in that total and spread fluids across the day. Daily weight checks can help notice fluid build\u2011up early.",
  "source_labels": [
    "HealthHub \u2014 Heart Failure: Monitoring Fluid and Salt Intake"
  ],
  "source_urls": [
    "https://www.healthhub.sg/health-conditions/heart-failure-monitoring-fluid-and-salt-intake"
  ],
  "clarifying_questions": [

## Cell 4 — Inspect your grounded agent in the Foundry portal (optional pause)

Your `carepal-jb` agent now has a **knowledge base attached** — go see it before cleanup.

1. Open **https://ai.azure.com** → project **`research-workshop`** → **Agents** → **`carepal-jb`**.
2. Check the **Knowledge / Tools** section — you'll see the file‑search index (vector store) you just built from code.
3. In the **Playground**, ask the diet question and watch the reply come back **with HealthHub citations** — the same grounding you validated above.

> ℹ️ **No conflict if you re-run:** re-running the cell above builds a fresh vector store and creates a **new version** of `carepal-jb`. Your name is unique in the shared project, so you never clash with other participants.


In [4]:
# 🧹 Run this LAST — once you've finished inspecting the agent in the portal.
# Deletes the agent version so the shared project stays tidy.
cleanup(agent)
print(f"Cleaned up {agent.name} ✅")


Cleaned up carepal-jb ✅
